# Credora Finance — Feature Engineering

## Objective

Prepare a clean loan-application-level dataset for credit default prediction by:

- Creating relevant engineered features
- Aggregating transaction data at customer level
- Integrating customer, credit, loan, and transaction data
- Keeping only loans with known default outcomes
- Removing data leakage and non-predictive identifiers
- Preparing the final dataset for machine learning

In [2]:
import pandas as pd
import numpy as np

In [8]:
customers = pd.read_csv('/content/customers.csv')
credit = pd.read_csv('/content/credit_history.csv')
loans = pd.read_csv('/content/loan_applications.csv')
transactions = pd.read_csv('/content/transactions.csv')

print("Datasets loaded successfully.")
print("-" * 45)
print("Customers:", customers.shape)
print("Credit History:", credit.shape)
print("Loan Applications:", loans.shape)
print("Transactions:", transactions.shape)

Datasets loaded successfully.
---------------------------------------------
Customers: (5000, 20)
Credit History: (5000, 14)
Loan Applications: (7500, 17)
Transactions: (40079, 8)


## 1. Date Preparation

In [9]:
# Convert date columns to datetime

customers['date_of_birth'] = pd.to_datetime(customers['date_of_birth'])
customers['customer_since'] = pd.to_datetime(customers['customer_since'])

credit['as_of_date'] = pd.to_datetime(credit['as_of_date'])

loans['application_date'] = pd.to_datetime(loans['application_date'])

transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

print("Date columns converted successfully.")
print("-" * 45)

print("Customers:")
print(customers[['date_of_birth', 'customer_since']].dtypes)

print("\nCredit History:")
print(credit['as_of_date'].dtype)

print("\nLoan Applications:")
print(loans['application_date'].dtype)

print("\nTransactions:")
print(transactions['transaction_date'].dtype)# Convert date columns to datetime

customers['date_of_birth'] = pd.to_datetime(customers['date_of_birth'])
customers['customer_since'] = pd.to_datetime(customers['customer_since'])

credit['as_of_date'] = pd.to_datetime(credit['as_of_date'])

loans['application_date'] = pd.to_datetime(loans['application_date'])

transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

print("Date columns converted successfully.")
print("-" * 45)

print("Customers:")
print(customers[['date_of_birth', 'customer_since']].dtypes)

print("\nCredit History:")
print(credit['as_of_date'].dtype)

print("\nLoan Applications:")
print(loans['application_date'].dtype)

print("\nTransactions:")
print(transactions['transaction_date'].dtype)

Date columns converted successfully.
---------------------------------------------
Customers:
date_of_birth     datetime64[ns]
customer_since    datetime64[ns]
dtype: object

Credit History:
datetime64[ns]

Loan Applications:
datetime64[ns]

Transactions:
datetime64[ns]
Date columns converted successfully.
---------------------------------------------
Customers:
date_of_birth     datetime64[ns]
customer_since    datetime64[ns]
dtype: object

Credit History:
datetime64[ns]

Loan Applications:
datetime64[ns]

Transactions:
datetime64[ns]


## 2. Customer-Level Feature Engineering

In [10]:
# Check customer_since date range

print("Customer Since Date Range")
print("-" * 40)
print("Earliest:", customers['customer_since'].min())
print("Latest:", customers['customer_since'].max())

print("\nMissing customer_since values:",
      customers['customer_since'].isna().sum())

Customer Since Date Range
----------------------------------------
Earliest: 2013-07-01 00:00:00
Latest: 2024-06-30 00:00:00

Missing customer_since values: 0


In [11]:
# Create debt-to-limit ratio

credit['debt_to_limit'] = (
    credit['total_outstanding_debt'] /
    credit['total_credit_limit']
)

print("Debt-to-limit feature created.")
print("-" * 40)

print(credit['debt_to_limit'].describe())

print("\nMissing values:",
      credit['debt_to_limit'].isna().sum())

print("Infinite values:",
      np.isinf(credit['debt_to_limit']).sum())

Debt-to-limit feature created.
----------------------------------------
count    5000.000000
mean        0.368042
std         0.187697
min         0.019718
25%         0.221962
50%         0.353032
75%         0.502025
max         0.989939
Name: debt_to_limit, dtype: float64

Missing values: 0
Infinite values: 0


## 3. Loan-Level Feature Engineering

In [12]:
# Add annual income to each loan application

loans_fe = loans.merge(
    customers[['customer_id', 'annual_income']],
    on='customer_id',
    how='left'
)

# Create loan-to-income ratio
loans_fe['loan_to_income'] = (
    loans_fe['requested_amount'] /
    loans_fe['annual_income']
)

print("Loan-to-income feature created.")
print("-" * 40)
print("Shape:", loans_fe.shape)

print("\nLoan-to-income summary:")
print(loans_fe['loan_to_income'].describe())

print("\nMissing values:",
      loans_fe['loan_to_income'].isna().sum())

print("Infinite values:",
      np.isinf(loans_fe['loan_to_income']).sum())

Loan-to-income feature created.
----------------------------------------
Shape: (7500, 19)

Loan-to-income summary:
count    7500.000000
mean        6.311876
std         9.873781
min         0.021475
25%         1.169429
50%         2.772443
75%         7.046149
max        99.975000
Name: loan_to_income, dtype: float64

Missing values: 0
Infinite values: 0


## 4. Transaction Feature Engineering

In [13]:
# Create transaction-level components for aggregation

transactions['inflow'] = np.where(
    transactions['transaction_type'] == 'Credit',
    transactions['amount'],
    0
)

transactions['outflow'] = np.where(
    transactions['transaction_type'] == 'Debit',
    transactions['amount'],
    0
)

transactions['emi_outflow'] = np.where(
    (transactions['transaction_type'] == 'Debit') &
    (transactions['category'] == 'EMI'),
    transactions['amount'],
    0
)

print("Transaction components created successfully.")
print("-" * 45)

print("Total inflow:", transactions['inflow'].sum())
print("Total outflow:", transactions['outflow'].sum())
print("Total EMI outflow:", transactions['emi_outflow'].sum())

Transaction components created successfully.
---------------------------------------------
Total inflow: 296169493.06
Total outflow: 266568008.17999998
Total EMI outflow: 41828625.550000004


In [14]:
# Aggregate transaction features to one row per customer

transaction_features = (
    transactions.groupby('customer_id')
    .agg(
        total_inflow_6m=('inflow', 'sum'),
        total_outflow_6m=('outflow', 'sum'),
        emi_outflow_6m=('emi_outflow', 'sum'),
        avg_balance=('balance_after', 'mean'),
        transaction_count=('transaction_id', 'count')
    )
    .reset_index()
)

# Create additional cash-flow features
transaction_features['net_cash_flow_6m'] = (
    transaction_features['total_inflow_6m'] -
    transaction_features['total_outflow_6m']
)

transaction_features['inflow_outflow_ratio'] = (
    transaction_features['total_inflow_6m'] /
    transaction_features['total_outflow_6m'].replace(0, np.nan)
)

print("Customer-level transaction features created.")
print("-" * 45)
print("Shape:", transaction_features.shape)
print("Unique customers:",
      transaction_features['customer_id'].nunique())

print("\nMissing values:")
print(transaction_features.isnull().sum())

Customer-level transaction features created.
---------------------------------------------
Shape: (5000, 8)
Unique customers: 5000

Missing values:
customer_id             0
total_inflow_6m         0
total_outflow_6m        0
emi_outflow_6m          0
avg_balance             0
transaction_count       0
net_cash_flow_6m        0
inflow_outflow_ratio    2
dtype: int64


In [15]:
# Handle undefined inflow/outflow ratios caused by zero outflow

max_finite_ratio = transaction_features[
    np.isfinite(transaction_features['inflow_outflow_ratio'])
]['inflow_outflow_ratio'].max()

transaction_features['inflow_outflow_ratio'] = (
    transaction_features['inflow_outflow_ratio']
    .fillna(max_finite_ratio)
)

print("Undefined inflow/outflow ratios handled.")
print("-" * 45)
print("Replacement value:", max_finite_ratio)

print("\nMissing values:",
      transaction_features['inflow_outflow_ratio'].isna().sum())

print("Infinite values:",
      np.isinf(transaction_features['inflow_outflow_ratio']).sum())

Undefined inflow/outflow ratios handled.
---------------------------------------------
Replacement value: 168.6398490837226

Missing values: 0
Infinite values: 0


## 5. Dataset Integration

In [16]:
# Create loan-application-level master dataset

master_df = (
    loans_fe
    .merge(
        customers.drop(columns=['annual_income']),
        on='customer_id',
        how='left'
    )
    .merge(
        credit,
        on='customer_id',
        how='left'
    )
    .merge(
        transaction_features,
        on='customer_id',
        how='left'
    )
)

print("Master dataset created successfully.")
print("-" * 45)

print("Original loan applications:", len(loans))
print("Master dataset rows:", len(master_df))
print("Master dataset columns:", master_df.shape[1])

print("\nDuplicate application IDs:",
      master_df['application_id'].duplicated().sum())

print("Missing customer IDs:",
      master_df['customer_id'].isna().sum())

Master dataset created successfully.
---------------------------------------------
Original loan applications: 7500
Master dataset rows: 7500
Master dataset columns: 58

Duplicate application IDs: 0
Missing customer IDs: 0


## 6. Modeling Population Selection

In [17]:
# Keep only loan applications with known default outcomes

model_df = master_df[
    master_df['loan_default'].notna()
].copy()

# Convert target to integer
model_df['loan_default'] = model_df['loan_default'].astype(int)

print("Modeling population created.")
print("-" * 45)

print("Master dataset shape:", master_df.shape)
print("Modeling dataset shape:", model_df.shape)

print("\nRows excluded due to unknown target:",
      len(master_df) - len(model_df))

print("\nTarget distribution:")
print(model_df['loan_default'].value_counts())

print("\nTarget distribution (%):")
print(
    model_df['loan_default']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Modeling population created.
---------------------------------------------
Master dataset shape: (7500, 58)
Modeling dataset shape: (3403, 58)

Rows excluded due to unknown target: 4097

Target distribution:
loan_default
0    2606
1     797
Name: count, dtype: int64

Target distribution (%):
loan_default
0    76.58
1    23.42
Name: proportion, dtype: float64


## 7. Data Leakage Removal

In [18]:
# Remove post-decision variables that would cause data leakage

leakage_columns = [
    'approval_status',
    'approved_amount',
    'decision_date',
    'disbursed_flag'
]

model_df = model_df.drop(columns=leakage_columns)

print("Leakage columns removed successfully.")
print("-" * 45)

print("Removed columns:")
for col in leakage_columns:
    print("-", col)

print("\nModeling dataset shape:", model_df.shape)

print("\nLeakage columns still present:")
print([col for col in leakage_columns if col in model_df.columns])

Leakage columns removed successfully.
---------------------------------------------
Removed columns:
- approval_status
- approved_amount
- decision_date
- disbursed_flag

Modeling dataset shape: (3403, 54)

Leakage columns still present:
[]


In [19]:
# Inspect remaining columns before feature selection

print("Total columns:", model_df.shape[1])
print("-" * 50)

for i, col in enumerate(model_df.columns, start=1):
    print(f"{i}. {col}")

Total columns: 54
--------------------------------------------------
1. application_id
2. customer_id
3. application_date
4. loan_type
5. loan_purpose
6. requested_amount
7. loan_term_months
8. interest_rate
9. applicant_monthly_income
10. existing_emi
11. debt_to_income_ratio
12. collateral_flag
13. loan_default
14. annual_income
15. loan_to_income
16. first_name
17. last_name
18. gender
19. date_of_birth
20. age
21. marital_status
22. dependents
23. education
24. employment_type
25. occupation
26. city
27. state
28. residence_type
29. years_at_residence
30. email
31. phone
32. kyc_status
33. customer_since
34. credit_id
35. as_of_date
36. credit_score
37. num_open_accounts
38. num_credit_inquiries_6m
39. credit_utilization_ratio
40. num_late_payments_30d
41. num_late_payments_90d
42. num_defaults_prior
43. bankruptcies
44. total_credit_limit
45. total_outstanding_debt
46. oldest_account_age_months
47. debt_to_limit
48. total_inflow_6m
49. total_outflow_6m
50. emi_outflow_6m
51. avg_b

In [20]:
# Check data types and missing values

column_check = pd.DataFrame({
    'Data_Type': model_df.dtypes,
    'Missing_Values': model_df.isnull().sum(),
    'Unique_Values': model_df.nunique()
})

print(column_check.to_string())

                                Data_Type  Missing_Values  Unique_Values
application_id                     object               0           3403
customer_id                        object               0           2309
application_date           datetime64[ns]               0            725
loan_type                          object               0              6
loan_purpose                       object               0             19
requested_amount                    int64               0           2003
loan_term_months                    int64               0              9
interest_rate                     float64               0           1282
applicant_monthly_income            int64               0            958
existing_emi                        int64               0            418
debt_to_income_ratio              float64               0           1488
collateral_flag                     int64               0              2
loan_default                        int64          

## 8. Customer Tenure Feature

In [21]:
# Calculate customer tenure at the time of loan application

model_df['customer_tenure_years'] = (
    (model_df['application_date'] - model_df['customer_since']).dt.days / 365.25
)

print("Customer tenure feature created.")
print("-" * 45)

print(model_df['customer_tenure_years'].describe())

print("\nMissing values:",
      model_df['customer_tenure_years'].isna().sum())

print("Negative tenure values:",
      (model_df['customer_tenure_years'] < 0).sum())

Customer tenure feature created.
---------------------------------------------
count    3403.000000
mean        4.049200
std         3.218591
min        -1.935661
25%         1.292266
50%         3.947981
75%         6.666667
max        10.888433
Name: customer_tenure_years, dtype: float64

Missing values: 0
Negative tenure values: 418


In [22]:
# Investigate rows with negative customer tenure

negative_tenure = model_df[
    model_df['customer_tenure_years'] < 0
][[
    'application_id',
    'customer_id',
    'application_date',
    'customer_since',
    'customer_tenure_years'
]]

print("Negative tenure rows:", len(negative_tenure))
print("-" * 60)

print(negative_tenure.head(20).to_string(index=False))

print("\nApplication date range for negative tenure:")
print("Earliest:", negative_tenure['application_date'].min())
print("Latest:", negative_tenure['application_date'].max())

print("\nCustomer-since date range for negative tenure:")
print("Earliest:", negative_tenure['customer_since'].min())
print("Latest:", negative_tenure['customer_since'].max())

Negative tenure rows: 418
------------------------------------------------------------
application_id customer_id application_date customer_since  customer_tenure_years
     APP800037  CUST100392       2023-03-09     2023-04-30              -0.142368
     APP800069  CUST104482       2022-10-03     2022-11-24              -0.142368
     APP800079  CUST100932       2022-12-22     2023-07-17              -0.566735
     APP800112  CUST102075       2022-10-24     2023-04-09              -0.457221
     APP800116  CUST104551       2023-03-15     2024-02-20              -0.936345
     APP800150  CUST101003       2024-01-05     2024-01-22              -0.046543
     APP800152  CUST103324       2024-05-01     2024-05-30              -0.079398
     APP800161  CUST102639       2022-09-12     2023-05-14              -0.668036
     APP800165  CUST104113       2022-07-22     2023-09-27              -1.182752
     APP800166  CUST100511       2023-07-19     2024-05-14              -0.821355
     APP800

In [23]:
# Remove customer tenure feature due to inconsistent source dates

model_df = model_df.drop(columns=['customer_tenure_years'])

print("Customer tenure feature removed.")
print("Current modeling dataset shape:", model_df.shape)

Customer tenure feature removed.
Current modeling dataset shape: (3403, 54)


## 9. Final Feature Selection

In [24]:
# Remove identifiers, personal information, raw dates,
# and redundant features

columns_to_drop = [
    # Identifiers
    'application_id',
    'customer_id',
    'credit_id',

    # Personal information
    'first_name',
    'last_name',
    'email',
    'phone',

    # Raw date columns
    'application_date',
    'date_of_birth',
    'customer_since',
    'as_of_date',

    # Redundant feature
    'credit_utilization_ratio'
]

model_df = model_df.drop(columns=columns_to_drop)

print("Non-modeling and redundant columns removed.")
print("-" * 50)

print("Columns removed:", len(columns_to_drop))
print("Final dataset shape:", model_df.shape)

print("\nRemaining columns:")
for i, col in enumerate(model_df.columns, 1):
    print(f"{i}. {col}")

Non-modeling and redundant columns removed.
--------------------------------------------------
Columns removed: 12
Final dataset shape: (3403, 42)

Remaining columns:
1. loan_type
2. loan_purpose
3. requested_amount
4. loan_term_months
5. interest_rate
6. applicant_monthly_income
7. existing_emi
8. debt_to_income_ratio
9. collateral_flag
10. loan_default
11. annual_income
12. loan_to_income
13. gender
14. age
15. marital_status
16. dependents
17. education
18. employment_type
19. occupation
20. city
21. state
22. residence_type
23. years_at_residence
24. kyc_status
25. credit_score
26. num_open_accounts
27. num_credit_inquiries_6m
28. num_late_payments_30d
29. num_late_payments_90d
30. num_defaults_prior
31. bankruptcies
32. total_credit_limit
33. total_outstanding_debt
34. oldest_account_age_months
35. debt_to_limit
36. total_inflow_6m
37. total_outflow_6m
38. emi_outflow_6m
39. avg_balance
40. transaction_count
41. net_cash_flow_6m
42. inflow_outflow_ratio


In [25]:
# Final validation of feature-engineered dataset

print("FINAL DATASET VALIDATION")
print("=" * 50)

print("Shape:", model_df.shape)

print("\nMissing values:", model_df.isnull().sum().sum())

# Check infinity only in numeric columns
numeric_cols = model_df.select_dtypes(include=np.number).columns

print(
    "Infinite values:",
    np.isinf(model_df[numeric_cols]).sum().sum()
)

print("\nDuplicate rows:", model_df.duplicated().sum())

print("\nTarget distribution:")
print(model_df['loan_default'].value_counts())

print("\nData types:")
print(model_df.dtypes.value_counts())

FINAL DATASET VALIDATION
Shape: (3403, 42)

Missing values: 0
Infinite values: 0

Duplicate rows: 0

Target distribution:
loan_default
0    2606
1     797
Name: count, dtype: int64

Data types:
int64      21
object     11
float64    10
Name: count, dtype: int64


In [26]:
# Inspect categorical features before saving

categorical_cols = model_df.select_dtypes(include='object').columns.tolist()

print("Categorical columns:", len(categorical_cols))
print("-" * 50)

for col in categorical_cols:
    print(f"{col}: {model_df[col].nunique()} unique values")

Categorical columns: 11
--------------------------------------------------
loan_type: 6 unique values
loan_purpose: 19 unique values
gender: 2 unique values
marital_status: 4 unique values
education: 5 unique values
employment_type: 5 unique values
occupation: 25 unique values
city: 20 unique values
state: 13 unique values
residence_type: 4 unique values
kyc_status: 2 unique values


## 10. Save Feature-Engineered Dataset

In [27]:
# Save the final feature-engineered dataset

output_path = '/content/credora_modeling_dataset.csv'

model_df.to_csv(output_path, index=False)

print("Feature-engineered dataset saved successfully.")
print("-" * 50)
print("File:", output_path)
print("Rows:", model_df.shape[0])
print("Columns:", model_df.shape[1])

Feature-engineered dataset saved successfully.
--------------------------------------------------
File: /content/credora_modeling_dataset.csv
Rows: 3403
Columns: 42


## 11. Feature Engineering Summary

The feature engineering stage prepared a clean loan-application-level dataset for credit default modeling.

Key steps completed:

- Converted relevant date columns to datetime format.
- Created `debt_to_limit` from total outstanding debt and total credit limit.
- Created `loan_to_income` from requested loan amount and annual income.
- Aggregated transaction data to customer level using six-month inflow, outflow, EMI outflow, average balance, transaction count, net cash flow, and inflow-to-outflow ratio.
- Handled undefined inflow-to-outflow ratios caused by customers with zero outflow.
- Integrated loan application, customer, credit history, and transaction information without duplicating loan applications.
- Restricted the modeling population to 3,403 loan applications with known `loan_default` outcomes.
- Removed post-decision variables that could cause data leakage.
- Investigated inconsistent customer tenure values and excluded the unreliable tenure feature.
- Removed identifiers, personally identifiable information, raw date fields, and the redundant `credit_utilization_ratio` feature.
- Produced a final modeling dataset containing 3,403 rows and 42 columns with no missing values, infinite values, or duplicate rows.

The final dataset is ready for preprocessing, train-test splitting, model training, and evaluation.